# Grouping and Pivoting Salary Data

Use Pandas groupby, agg, and pivot_table to compare salary patterns across the dataset's major categories. This notebook uses the same cleaned rows and derived growth and spread measures as the earlier analyses. Run from Day 72/notebooks.


In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/salaries_by_college_major.csv").dropna().copy()
df["Spread"] = (
    df["Mid-Career 90th Percentile Salary"]
    - df["Mid-Career 10th Percentile Salary"]
)
df["Growth %"] = (
    (df["Mid-Career Median Salary"] - df["Starting Median Salary"])
    / df["Starting Median Salary"] * 100
)
display(df.head())


## 1. Aggregate salary measures by group

Compare the count of majors and the mean, median, minimum, and maximum starting and mid-career median salaries for each group.


In [ ]:
group_summary = df.groupby("Group").agg(
    Major_Count=("Undergraduate Major", "count"),
    Starting_Mean=("Starting Median Salary", "mean"),
    Starting_Median=("Starting Median Salary", "median"),
    Mid_Career_Mean=("Mid-Career Median Salary", "mean"),
    Mid_Career_Median=("Mid-Career Median Salary", "median"),
    Average_Growth_Percent=("Growth %", "mean"),
    Average_Spread=("Spread", "mean"),
).sort_values("Mid_Career_Mean", ascending=False)
display(group_summary.round(2))


## 2. Find the top major in each group

Sort within each category to identify the major with the highest mid-career median salary.


In [ ]:
top_by_group = (
    df.sort_values("Mid-Career Median Salary", ascending=False)
      .groupby("Group", as_index=False)
      .first()[["Group", "Undergraduate Major", "Starting Median Salary", "Mid-Career Median Salary", "Growth %", "Spread"]]
)
display(top_by_group)


## 3. Pivot salaries by group

A pivot table shows average starting and mid-career salaries side by side for each category.


In [ ]:
salary_pivot = pd.pivot_table(
    df,
    index="Group",
    values=["Starting Median Salary", "Mid-Career Median Salary"],
    aggfunc="mean",
).sort_values("Mid-Career Median Salary", ascending=False)
display(salary_pivot.round(2))


## 4. Pivot individual majors

Display each major's mid-career median salary by group. Empty category combinations remain blank in the table.


In [ ]:
major_pivot = pd.pivot_table(
    df,
    index="Undergraduate Major",
    columns="Group",
    values="Mid-Career Median Salary",
    aggfunc="mean",
)
display(major_pivot)


## 5. Check the derived data

Review a few high growth and wide spread majors to connect the grouped summaries to individual records.


In [ ]:
display(df.nlargest(5, "Growth %")[
    ["Undergraduate Major", "Group", "Growth %", "Spread"]
])
display(df.nlargest(5, "Spread")[
    ["Undergraduate Major", "Group", "Growth %", "Spread"]
])
